# 07 · Foundry Agent Service → Bifrost
Foundry agents can use models **behind your own gateway** (bring-your-own-model). We register the APIM gateway as an `ApiManagement` connection in the Foundry project, expose the router alias `heimdal-auto`, and create an agent on it – every agent LLM call now flows through HEIMDAL (metrics, limits, routing).

> Uses preview APIs (`azure-ai-projects` 2.x, connection API). Adjust to the latest docs if a call signature changed.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

import json, subprocess, shutil, tempfile, os
acct, proj = env('HEIMDAL_FOUNDRY_ACCOUNT'), env('HEIMDAL_FOUNDRY_PROJECT')
url = (f"https://management.azure.com/subscriptions/{env('HEIMDAL_SUBSCRIPTION_ID')}/resourceGroups/{env('HEIMDAL_RESOURCE_GROUP')}"
       f"/providers/Microsoft.CognitiveServices/accounts/{acct}/projects/{proj}/connections/bifrost-apim?api-version=2025-04-01-preview")
body = {"properties": {
    "category": "ApiManagement", "target": env('HEIMDAL_LLM_BASE_URL'), "authType": "ApiKey",
    "credentials": {"key": user_key('black-widow')},      # the agent runs as 'black-widow' for token accounting
    "metadata": {"deploymentInPath": "false", "inferenceAPIVersion": "2024-02-01",
                 "models": json.dumps([{"name": "heimdal-auto", "properties": {"model": {"name": "heimdal-auto", "version": "", "format": "OpenAI"}}}])}}}
f = os.path.join(tempfile.gettempdir(), 'bifrost-conn.json'); json.dump(body, open(f, 'w'))
out = subprocess.run([shutil.which('az'), 'rest', '--method', 'put', '--url', url, '--body', '@' + f], capture_output=True, text=True)
print(out.stdout[:400] or out.stderr[:800])

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

project = AIProjectClient(endpoint=env('HEIMDAL_FOUNDRY_PROJECT_ENDPOINT'), credential=DefaultAzureCredential())
agent = project.agents.create_version(
    agent_name='heimdall-watcher',
    definition=PromptAgentDefinition(model='bifrost-apim/heimdal-auto',
                                     instructions='You are Heimdall, guardian of the Bifrost. Be concise and a little epic.'))
oai = project.get_openai_client()
resp = oai.responses.create(input='Who guards the Bifrost and why does that matter for an API gateway?',
                            extra_body={'agent_reference': {'name': agent.name, 'type': 'agent_reference'}})
print(resp.output_text)

Check notebook 02's KQL: the agent's calls are attributed to `black-widow` and routed by the same policy.